Richard Davies · **Automated Data Visualisation for Policymaking** · PP434 · 2026/27

# Menu prices: quick start

How much has the price of a latte risen? And is it really dearer in London?

The LSE Growth Lab collected **3.3 million photos of menus**, posted with Google Maps reviews across 25 UK cities, and used **large language models** (AI that can read images) to pull out the prices: **12.8 million prices** from 80,000 cafes, pubs and restaurants, from 2011 to 2026. It's part of the AutoCPI project, which uses web data to measure prices in more detail than official statistics can: in June 2026, the ONS collected just 115 hospitality prices for the whole of Wales.

This notebook starts with a **small summary** of that data: the median price of 12 well-known items, by year and region. (The **median** is the middle price: half are cheaper, half dearer.)

> **Before you start:** click **File → Save a copy in Drive**. Otherwise, your changes will not be saved.

In [ ]:
# Colab comes with an older version of Altair (v5). This updates it to the latest version (v6),
# so the charts we save match the Vega-Lite version used on our websites.
# Run this before importing Altair. (If you've already imported it, use Runtime → Restart session, then run again.)
# On your own computer (see our local Python guide), this just checks you have the latest version.
%pip install altair --upgrade --quiet

In [ ]:
import pandas as pd
import altair as alt

alt.data_transformers.disable_max_rows()     # Altair stops at 5,000 rows by default: our tables are bigger

DATA = "https://raw.githubusercontent.com/jhellingsdata/LSE-PP434/main/weeks/week08/data/"
MAPS = "https://raw.githubusercontent.com/jhellingsdata/map-data/refs/heads/main/gbr/"

SOURCE = "Source: LSE Growth Lab, AutoCPI (menu photos)"     # for chart subtitles

<br><br>

---

<br>

## 1. Load the summary

One row per item, year and region. `prices` is how many menu prices each median is based on: the summary of 270,000 prices fits in a 50 KB file.

In [ ]:
prices = pd.read_csv(DATA + "menu_iconic_by_year_region.csv")
prices.head()

In [ ]:
prices["item"].unique()

<br>

### Years as dates

`year` is a whole number (`2017`). For a time axis, Altair needs a **date**: as a number, Vega-Lite would read 2017 as 2,017 milliseconds after 1 January 1970. So we add a `date` column, 1 January of each year:

In [ ]:
prices["date"] = pd.to_datetime(prices["year"], format="%Y")     # 2017 → 2017-01-01
prices.head(3)

<br>

> **Two alternatives.** Text works too: `prices["year"].astype(str)` gives `"2017"`, which Vega-Lite also reads as a date. Or use `x="year:O"` (ordinal), which treats each year as a label, evenly spaced, like a category. That's fine for one bar per year, but a missing year wouldn't leave a gap.

<br><br>

---

<br>

## 2. The price of a latte

Keep the rows for a latte, across the whole UK (`itl1_id` is `"UK"`), and chart them with `date:T` on the x-axis:

In [ ]:
latte = prices[(prices["item"] == "latte") & (prices["itl1_id"] == "UK")]

alt.Chart(latte).mark_line(color="#8c5a2b", point={"color": "#8c5a2b"}).encode(
    x=alt.X("date:T").title(None),
    y=alt.Y("median_price:Q").title("£").scale(zero=False),
    tooltip=[
        alt.Tooltip("date:T").format("%Y").title("Year"),
        alt.Tooltip("median_price:Q").format(".2f").title("Median (£)"),
        alt.Tooltip("prices:Q").format(",").title("Prices"),
    ],
).properties(width=450, height=250,
             title=alt.Title("The price of a latte", subtitle="Median price on UK menus. " + SOURCE))

<br>

From £2.50 in 2017 to £4.00 in 2026 (the 2026 figure is for January to August). Hover over the points: in 2017 the median comes from about 1,500 prices; in 2025, from 23,000. More people photograph menus every year.

> **Try it:** change `"latte"` to `"fish & chips"` or `"guinness"`.

<br><br>

---

<br>

## 3. Which prices rose most?

Take each item's UK price in 2019 (before Covid) and in 2025. With the item as the index, pandas lines them up by item, so we can divide one by the other:

In [ ]:
uk = prices[prices["itl1_id"] == "UK"]

before = uk[uk["year"] == 2019].set_index("item")["median_price"]
after = uk[uk["year"] == 2025].set_index("item")["median_price"]

table = (after / before - 1).round(3).reset_index(name="change")
table.sort_values("change")

<br>

`change` is a share: `0.411` means 41.1%. Keep it that way, and let the chart show it as a percentage with `format(".0%")`, on the axis and in the tooltip:

In [ ]:
alt.Chart(table).mark_bar(color="#c8102e").encode(
    x=alt.X("change:Q").title("Change in median price, 2019 to 2025").axis(format=".0%"),
    y=alt.Y("item:N").sort("-x").title(None),
    tooltip=[
        alt.Tooltip("item:N").title("Item"),
        alt.Tooltip("change:Q").format(".0%").title("Change"),
    ],
).properties(width=400, title=alt.Title("Six years of eating-out inflation", subtitle=SOURCE))

<br><br>

---

<br>

## 4. Map it: is a latte dearer in London?

The regions are the UK's 12 **ITL1** regions, with the same codes as map-data's `ITL1_UK_2025.json` (`properties.id`). So the map is a choropleth with a lookup, as in Week 7, with a grey layer underneath for any region without data.

The map file is loaded with `alt.Data`, which matches Vega-Lite's `"data": {"url": ..., "format": {"type": "topojson", "feature": "geog"}}`:

In [ ]:
latte_2025 = prices[(prices["item"] == "latte") & (prices["year"] == 2025) & (prices["itl1_id"] != "UK")]

regions = alt.Data(url=MAPS + "ITL1_UK_2025.json",
                   format=alt.DataFormat(type="topojson", feature="geog"))

grey = alt.Chart(regions).mark_geoshape(fill="#e3e3e3", stroke="white")

coloured = alt.Chart(regions).transform_lookup(
    lookup="properties.id",
    from_=alt.LookupData(latte_2025, key="itl1_id", fields=["median_price", "prices"]),
).mark_geoshape(stroke="white").encode(
    color=alt.Color("median_price:Q").title("£").scale(scheme="oranges"),
    tooltip=[
        alt.Tooltip("properties.name:N").title("Region"),
        alt.Tooltip("median_price:Q").format(".2f").title("Median latte (£)"),
        alt.Tooltip("prices:Q").format(",").title("Prices"),
    ],
)

(grey + coloured).project("mercator").properties(width=320, height=420,
    title=alt.Title("A latte by region, 2025", subtitle=SOURCE))

<br>

London is dearest (£4.00) and Wales cheapest (£3.50), but most regions are within 10p of each other. The research team find the same for chains: **prices are set nationally**, not locally.

> **Careful:** the data comes from 25 cities, so each region's figure describes its cities (e.g. "East" is mostly Norwich), not the whole region.

> **Other ways to load a map.** You may see the shortcut `alt.topo_feature(url, "geog")` online: it makes the same `alt.Data`. For a GeoJSON file, use `alt.Data(url=..., format=alt.DataFormat(property="features"))`.

<br><br>

---

<br>

## <font color='green'>Try it</font>

**1.** Map another item, e.g. `"fish & chips"`, `"guinness"` or `"margherita"`. Is the pattern the same as for a latte?

**2.** Change the map to 2019. Did the gap between London and the rest grow or shrink?

**3.** Save your map with `.save("my_map.json")` and embed it on your website.

In [ ]:
# Your code here
